# Fraudfinder - Explainable AI Demo

## Overview

This series of labs are updated upon [FraudFinder](https://github.com/googlecloudplatform/fraudfinder) repository which builds a end-to-end real-time fraud detection system on Google Cloud. Throughout the FraudFinder labs, you will learn how to read historical bank transaction data stored in data warehouse, read from a live stream of new transactions, perform exploratory data analysis (EDA), do feature engineering, ingest features into a feature store, train a model using feature store, register your model in a model registry, evaluate your model, deploy your model to an endpoint, do real-time inference on your model with feature store, and monitor your model.


### Objective

This notebook demonstrates a critical step in a real-time fraud detection system: **making a prediction on a new, incoming transaction.** We'll simulate a real-world scenario where a transaction is received and we need to quickly determine if it's fraudulent.

To do this, we'll use a deployed machine learning model on a **Vertex AI Endpoint**. However, a model needs more than just the raw transaction data to make an accurate prediction; it needs **features**. These are the data points that give the model context about the transaction, the customer, and the terminal involved.

This is where the **Vertex AI Feature Store** comes in. The Feature Store provides low-latency access to pre-calculated features. In this notebook, we will:

1. **Simulate a new transaction** by pulling a sample transaction from a Pub/Sub topic.
2. **Enrich the transaction data** by fetching the latest customer and terminal features from the Vertex AI Feature Store.
3. **Construct a prediction request** with the combined transaction data and features.
4. **Send the request to the Vertex AI Endpoint** to get a real-time fraud prediction.

This entire process mirrors a production-level inference pipeline, showcasing how to leverage Vertex AI services to build a powerful and responsive fraud detection system.

### Load configuration settings from the setup notebook

Set the constants used in this notebook and load the config settings from the `00_environment_setup.ipynb` notebook.

In [ ]:
GCP_PROJECTS = !gcloud config get-value project
PROJECT_ID = GCP_PROJECTS[0]
BUCKET_NAME = f"{PROJECT_ID}-fraudfinder"
config = !gsutil cat gs://{BUCKET_NAME}/config/notebook_env.py
print(config.n)
exec(config.n)

#### Find latest fraud transaction:

In [ ]:
%%bigquery df_fraud_transaction
SELECT
    raw_tx.TX_ID AS tx_id,
    raw_tx.TX_TS AS tx_timestamp,
    raw_tx.CUSTOMER_ID AS customer_id,
    raw_tx.TERMINAL_ID AS terminal_id,
    raw_tx.TX_AMOUNT AS tx_amount,
    raw_lb.TX_FRAUD AS tx_fraud,
FROM
    `tx.ingestion_tx_records` AS raw_tx
LEFT JOIN
    `tx.ingestion_tx_labels` AS raw_lb
ON
    raw_tx.TX_ID = raw_lb.TX_ID
WHERE
    raw_lb.TX_FRAUD=1
ORDER BY tx_timestamp DESC
LIMIT 1

In [ ]:
df_fraud_transaction


In [ ]:
query_params = {
    "transaction_id": df_fraud_transaction['tx_id'][0]
}

In [ ]:
query_params

#### Using transaction ID to fetch features in point of time from BigQuery

In [ ]:
%%bigquery raw_transaction --params $query_params
SELECT
    raw_tx.TX_TS AS tx_timestamp,
    raw_tx.CUSTOMER_ID AS customer_id,
    raw_tx.TERMINAL_ID AS terminal_id,
    raw_tx.TX_AMOUNT AS tx_amount,
    raw_lb.TX_FRAUD AS tx_fraud,
FROM
    `tx.ingestion_tx_records` AS raw_tx
LEFT JOIN
    `tx.ingestion_tx_labels` AS raw_lb
ON
    raw_tx.TX_ID = raw_lb.TX_ID
WHERE
    raw_tx.TX_ID=@transaction_id

In [ ]:
raw_transaction

In [ ]:
%%bigquery transaction_with_features --params $query_params

DECLARE input_cust STRING DEFAULT @transaction_id;

WITH 
  # ---------------------------------------------------------
  # Create a single-row table from the inputs
  # ---------------------------------------------------------
  input_lookup AS (
    SELECT
        raw_tx.TX_TS AS feature_timestamp,
        raw_tx.CUSTOMER_ID customer_id,
        raw_tx.TERMINAL_ID AS terminal_id,
        raw_tx.TX_AMOUNT AS tx_amount
    FROM
        `tx.ingestion_tx_records` AS raw_tx
    WHERE
        raw_tx.TX_ID=@transaction_id
  ),
  customer_lookup AS (
    SELECT feature_timestamp AS `time`, customer_id AS entity_id FROM input_lookup
  ),
  terminal_lookup AS (
    SELECT feature_timestamp AS `time`, terminal_id AS entity_id FROM input_lookup
  )

SELECT
  input_lookup.tx_amount AS tx_amount,

  # --- Customer Batch Features ---
  IFNULL(f_customers.customer_id_avg_amount_1day_window, 0.0) as customer_id_avg_amount_1day_window,
  IFNULL(f_customers.customer_id_avg_amount_7day_window, 0.0) as customer_id_avg_amount_7day_window,
  IFNULL(f_customers.customer_id_avg_amount_14day_window, 0.0) as customer_id_avg_amount_14day_window,
  IFNULL(CAST(f_customers.customer_id_nb_tx_1day_window AS FLOAT64), 0.0) as customer_id_nb_tx_1day_window,
  IFNULL(CAST(f_customers.customer_id_nb_tx_7day_window AS FLOAT64), 0.0) as customer_id_nb_tx_7day_window,
  IFNULL(CAST(f_customers.customer_id_nb_tx_14day_window AS FLOAT64), 0.0) as customer_id_nb_tx_14day_window,

  # --- Terminal Batch Features ---
  IFNULL(f_terminals.terminal_id_risk_1day_window, 0.0)  AS terminal_id_risk_1day_window,
  IFNULL(f_terminals.terminal_id_risk_7day_window, 0.0)  AS terminal_id_risk_7day_window,
  IFNULL(f_terminals.terminal_id_risk_14day_window, 0.0)  AS terminal_id_risk_14day_window,
  IFNULL(CAST(f_terminals.terminal_id_nb_tx_1day_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_1day_window,
  IFNULL(CAST(f_terminals.terminal_id_nb_tx_7day_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_7day_window,
  IFNULL(CAST(f_terminals.terminal_id_nb_tx_14day_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_14day_window,

  # --- Customer Streaming Features ---
  IFNULL(f_cust_stream.customer_id_avg_amount_15min_window, 0.0) as customer_id_avg_amount_15min_window,
  IFNULL(f_cust_stream.customer_id_avg_amount_30min_window, 0.0) as customer_id_avg_amount_30min_window,
  IFNULL(f_cust_stream.customer_id_avg_amount_60min_window, 0.0) as customer_id_avg_amount_60min_window,
  IFNULL(CAST(f_cust_stream.customer_id_nb_tx_15min_window AS FLOAT64), 0.0)  AS customer_id_nb_tx_15min_window,
  IFNULL(CAST(f_cust_stream.customer_id_nb_tx_30min_window AS FLOAT64), 0.0)  AS customer_id_nb_tx_30min_window,
  IFNULL(CAST(f_cust_stream.customer_id_nb_tx_60min_window AS FLOAT64), 0.0)  AS customer_id_nb_tx_60min_window,

  # --- Terminal Streaming Features ---
  IFNULL(f_term_stream.terminal_id_avg_amount_15min_window, 0.0) as terminal_id_avg_amount_15min_window,
  IFNULL(f_term_stream.terminal_id_avg_amount_30min_window, 0.0) as terminal_id_avg_amount_30min_window,
  IFNULL(f_term_stream.terminal_id_avg_amount_60min_window, 0.0) as terminal_id_avg_amount_60min_window,
  IFNULL(CAST(f_term_stream.terminal_id_nb_tx_15min_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_15min_window,
  IFNULL(CAST(f_term_stream.terminal_id_nb_tx_30min_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_30min_window,
  IFNULL(CAST(f_term_stream.terminal_id_nb_tx_60min_window AS FLOAT64), 0.0)  AS terminal_id_nb_tx_60min_window

FROM input_lookup
LEFT JOIN ML.ENTITY_FEATURES_AT_TIME(
  TABLE `tx.t_customers_batch_features`,
  TABLE customer_lookup, num_rows => 1, ignore_feature_nulls => TRUE
) AS f_customers ON input_lookup.customer_id = f_customers.entity_id

LEFT JOIN ML.ENTITY_FEATURES_AT_TIME(
  TABLE `tx.t_terminals_batch_features`,
  TABLE terminal_lookup, num_rows => 1, ignore_feature_nulls => TRUE
) AS f_terminals ON input_lookup.terminal_id = f_terminals.entity_id

LEFT JOIN ML.ENTITY_FEATURES_AT_TIME(
  TABLE `tx.t_customers_streaming_features`,
  TABLE customer_lookup, num_rows => 1, ignore_feature_nulls => TRUE
) AS f_cust_stream ON input_lookup.customer_id = f_cust_stream.entity_id

LEFT JOIN ML.ENTITY_FEATURES_AT_TIME(
  TABLE `tx.t_terminals_streaming_features`,
  TABLE terminal_lookup, num_rows => 1, ignore_feature_nulls => TRUE
) AS f_term_stream ON input_lookup.terminal_id = f_term_stream.entity_id;

In [ ]:
transaction_with_features

In [ ]:
tx_fs_json = transaction_with_features.to_dict(orient='records')

In [ ]:
tx_fs_json

### Import libraries and define constants

#### Libraries
Here, we'll import the necessary libraries for this notebook.

In [ ]:
from google.cloud import aiplatform as vertex_ai

In [ ]:
# Vertex AI SDK
vertex_ai.init(project=PROJECT_ID, location=REGION, staging_bucket=BUCKET_NAME)

### Define helper methods

### Retrieve the Deployed Model Endpoint

Before we can get a prediction, we need to connect to our deployed model. In Vertex AI, deployed models are exposed as **Endpoints**. An endpoint provides a URL that you can send prediction requests to. The following code retrieves the endpoint for our trained fraud detection model. We filter by the `display_name` to ensure we get the correct one.

In [ ]:
endpoints = vertex_ai.Endpoint.list(
    filter=f"display_name={ENDPOINT_NAME}",  # optional: filter by specific endpoint name
    order_by="update_time",
)

ENDPOINT_ID = endpoints[-1].name
print(ENDPOINT_ID)

In [ ]:
print(ENDPOINT_ID)
from google.cloud import aiplatform as aiplatform

# Instantiate Vertex AI Endpoint object
endpoint_obj = aiplatform.Endpoint(ENDPOINT_ID)

### Simulate an Incoming Transaction

In a real-world system, new transactions would be continuously streaming in. To simulate this, we'll read a single transaction from the `ff-tx-sub` Pub/Sub subscription. This subscription receives the same raw transaction data that our real-time feature engineering pipeline processes. The `read_from_sub` helper function will pull one message for us to use in our prediction request.


### Putting It All Together: Building the Prediction Request

Now we'll perform the full, end-to-end process for a single transaction:

1.  **Get a transaction:** We'll use the transaction we read from Pub/Sub earlier.
2.  **Set default feature values:** We start with a dictionary of default values for all our features. This is a good practice to ensure the model receives a complete feature vector, even if a feature lookup fails.
3.  **Add transaction amount:** We add the `TX_AMOUNT` from our simulated transaction to the payload.
4.  **Lookup and add customer features:**
5.  **Lookup and add terminal features:**
6.  **Send for prediction:** Finally, we send the complete payload to our model endpoint's `predict` method.

The result will be a real-time fraud prediction for our incoming transaction, enriched with the latest features from our BigQuery historical data.

In [ ]:
import pprint

pp = pprint.PrettyPrinter(compact=True)

payload=tx_fs_json

print("-------------------------------------------------------")
print("[Payload to be sent to Vertex AI endpoint]")
pp.pprint(payload)
print("-------------------------------------------------------")

result = endpoint_obj.explain(instances=payload)

print("-------------------------------------------------------")
pp.pprint(f"[Prediction result]: {result}")
print("-------------------------------------------------------")
pp.pprint(result.explanations)

### END

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.